In [1]:
# ==============================================================================
# Project: Cybercrime & Fraud NLP Classification Model
# Dataset: Hugging Face (Dev523/Crime-Reports-Dataset) ~93.7k Rows
# Target: Multi-class Text Categorization (crimeaditionalinfo -> category)
# Expected Training Time: ~25 - 45 seconds on Google Colab CPU
# ==============================================================================

import time
import re
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

# ------------------------------------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------------------------------------
print("[Step 1/5] Downloading and loading dataset...")
dataset_url = "https://huggingface.co/datasets/Dev523/Crime-Reports-Dataset/resolve/main/train.csv"

# Load directly from URL (or replace with local path if uploaded to Colab)
df = pd.read_csv(dataset_url)
print(f"-> Successfully loaded {len(df):,} raw records.")

# ------------------------------------------------------------------------------
# 2. Text Preprocessing & Cleaning
# ------------------------------------------------------------------------------
print("\n[Step 2/5] Cleaning and filtering incident narratives...")
class_counts = df['category'].value_counts()
valid_categories = class_counts[class_counts >= 2].index
df = df[df['category'].isin(valid_categories)]
def clean_text(text: str) -> str:
    """Cleans raw citizen complaints for NLP modeling."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)         # Remove special characters
    return re.sub(r'\s+', ' ', text).strip()            # Normalize whitespace

# Drop rows with missing target or input values
df = df.dropna(subset=['crimeaditionalinfo', 'category'])

# Apply text cleaning
df['clean_text'] = df['crimeaditionalinfo'].apply(clean_text)

# Remove noisy, extremely short entries (< 3 words)
df = df[df['clean_text'].apply(lambda x: len(x.split()) >= 3)]

print(f"-> Filtered down to {len(df):,} high-quality text records.")
print("\nCategory Distribution:")
print(df['category'].value_counts())

# ------------------------------------------------------------------------------
# 3. Stratified Train / Test Split (80% Train, 20% Test)
# ------------------------------------------------------------------------------
print("\n[Step 3/5] Splitting dataset into training (80%) and testing (20%) sets...")

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['category'],
    test_size=0.20,
    random_state=42,
    stratify=df['category']
)

print(f"-> Training samples : {len(X_train):,}")
print(f"-> Testing samples  : {len(X_test):,}")

# ------------------------------------------------------------------------------
# 4. Feature Extraction (TF-IDF) & Model Training
# ------------------------------------------------------------------------------
print("\n[Step 4/5] Extracting TF-IDF features and training the classifier...")
start_time = time.time()

# Configure TF-IDF with unigrams + bigrams and sublinear term scaling
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=25000,
    sublinear_tf=True,
    min_df=2,
    stop_words='english'
)

# Transform text to feature matrices
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Logistic Regression with balanced class weights and multi-threading
base_model = LogisticRegression(
    C=3.0,
    max_iter=1000,
    class_weight='balanced',
    solver='saga',
    n_jobs=-1,
    random_state=42
)

# Wrap in CalibratedClassifierCV to obtain calibrated prediction probabilities
calibrated_model = CalibratedClassifierCV(estimator=base_model, method='sigmoid', cv=3)
calibrated_model.fit(X_train_vec, y_train)

elapsed_time = time.time() - start_time
print(f"-> Model training completed in {elapsed_time:.2f} seconds!")

# ------------------------------------------------------------------------------
# 5. Model Evaluation & Performance Metrics
# ------------------------------------------------------------------------------
print("\n[Step 5/5] Evaluating model performance on the test set...")

y_pred = calibrated_model.predict(X_test_vec)
overall_acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print("=" * 65)
print(f"  OVERALL ACCURACY : {overall_acc * 100:.2f}%")
print(f"  MACRO F1-SCORE   : {macro_f1 * 100:.2f}%")
print("=" * 65)

print("\nDetailed Per-Category Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

# ------------------------------------------------------------------------------
# 6. Save Trained Artifacts for Backend Integration
# ------------------------------------------------------------------------------
print("\nSaving trained model artifacts...")
joblib.dump(calibrated_model, 'cybercrime_classifier.joblib')
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')


print("\n[SUCCESS] Artifacts exported:")
print("  1. 'cybercrime_classifier.joblib' (Trained model)")
print("  2. 'tfidf_vectorizer.joblib'      (Vocabulary & feature extractor)")
print("\nDownload these two files from the Colab left sidebar (Files tab) to use in your backend!")

[Step 1/5] Downloading and loading dataset...
-> Successfully loaded 93,686 raw records.

[Step 2/5] Cleaning and filtering incident narratives...
-> Filtered down to 88,768 high-quality text records.

Category Distribution:
category
Online Financial Fraud                                  53324
Online and Social Media Related Crime                   11752
Any Other Cyber Crime                                   10619
Cyber Attack/ Dependent Crimes                           3608
RapeGang Rape RGRSexually Abusive Content                2816
Sexually Obscene material                                1797
Hacking  Damage to computercomputer system etc           1665
Sexually Explicit Act                                    1515
Cryptocurrency Crime                                      476
Online Gambling  Betting                                  435
Child Pornography CPChild Sexual Abuse Material CSAM      370
Online Cyber Trafficking                                  175
Cyber Terrorism       

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


-> Model training completed in 616.16 seconds!

[Step 5/5] Evaluating model performance on the test set...
  OVERALL ACCURACY : 74.12%
  MACRO F1-SCORE   : 37.47%

Detailed Per-Category Classification Report:
                                                      precision    recall  f1-score   support

                               Any Other Cyber Crime     0.5769    0.0141    0.0276      2124
Child Pornography CPChild Sexual Abuse Material CSAM     0.6296    0.2297    0.3366        74
                                Cryptocurrency Crime     0.6154    0.5053    0.5549        95
                      Cyber Attack/ Dependent Crimes     1.0000    1.0000    1.0000       722
                                     Cyber Terrorism     0.0000    0.0000    0.0000        32
      Hacking  Damage to computercomputer system etc     0.4531    0.2613    0.3314       333
                            Online Cyber Trafficking     0.0000    0.0000    0.0000        35
                              Online F

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



[SUCCESS] Artifacts exported:
  1. 'cybercrime_classifier.joblib' (Trained model)
  2. 'tfidf_vectorizer.joblib'      (Vocabulary & feature extractor)

Download these two files from the Colab left sidebar (Files tab) to use in your backend!
